# LeetCode 1740 — Find Distance in a Binary Tree bằng Python

Link đề: https://leetcode.com/problems/find-distance-in-a-binary-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Breadth-First Search, Hash Table, Binary Tree, Lowest Common Ancestor

> **Bài Premium.** Đề bài trong notebook được viết lại theo hiểu biết về bài (nguồn: bản mô tả của doocs/leetcode). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc trên LeetCode.

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def findDistance(self, root: Optional[TreeNode], p: int, q: int) -> int:
        ...
```

Lộ trình: **chuyển cây thành đồ thị vô hướng + BFS** từ `p` tới `q` → **hai đường đi gốc→nút**, bỏ phần chung (`len + len − 2·chung`) → **con trỏ cha + độ sâu**, leo lên tới **tổ tiên chung thấp nhất (LCA)** → **một lượt hậu thứ tự** (post-order) trả về khoảng cách ngay khi gặp LCA. Tất cả viết **không đệ quy** vì cây có thể suy biến thành đường thẳng `10⁴` nút.

## 1. Mục tiêu học

1. Tự cài `TreeNode`, hàm dựng cây từ mảng level-order kiểu LeetCode (có `None`) và tìm nút theo giá trị.
2. Hiểu công thức nền tảng: `dist(p, q) = depth(p) + depth(q) − 2·depth(LCA(p, q))`.
3. Tìm **LCA** bằng ba cách: so sánh đường đi gốc→nút, leo con trỏ cha theo độ sâu, và hậu thứ tự.
4. Viết DFS/BFS **lặp** để tránh `RecursionError` với cây sâu `10⁴`.
5. Kiểm chứng bằng **Floyd–Warshall** (khoảng cách mọi cặp) trên cây nhỏ — một cách độc lập hoàn toàn.

## 2. Đề bài & ví dụ

Cho gốc `root` của một cây nhị phân và hai số nguyên `p`, `q`. Trả về **khoảng cách** giữa nút có giá trị `p` và nút có giá trị `q`.

Khoảng cách giữa hai nút là **số cạnh** trên đường đi từ nút này tới nút kia.

**Ví dụ 1:** `root = [3,5,1,6,2,0,8,null,null,7,4]`, `p = 5`, `q = 0` → `3` (đường `5 − 3 − 1 − 0`).

**Ví dụ 2:** cùng cây, `p = 5`, `q = 7` → `2` (đường `5 − 2 − 7`).

**Ví dụ 3:** cùng cây, `p = 5`, `q = 5` → `0`.

**Ràng buộc:**
- Số nút trong `[1, 10⁴]`
- `0 ≤ Node.val ≤ 10⁹`, các giá trị **đôi một khác nhau**
- `p`, `q` là giá trị có trong cây.

In [1]:
import random
import time
from collections import deque
from typing import List, Optional

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right
    def __repr__(self):
        return f"TreeNode({self.val})"

def build_tree(arr):
    """Dựng cây từ mảng level-order kiểu LeetCode (None = không có nút). Không đệ quy."""
    if not arr or arr[0] is None:
        return None
    root = TreeNode(arr[0])
    q = deque([root]); i = 1
    while q and i < len(arr):
        node = q.popleft()
        if i < len(arr) and arr[i] is not None:
            node.left = TreeNode(arr[i]); q.append(node.left)
        i += 1
        if i < len(arr) and arr[i] is not None:
            node.right = TreeNode(arr[i]); q.append(node.right)
        i += 1
    return root

def to_level_list(root):
    """Xuất cây ra mảng level-order kiểu LeetCode (bỏ None thừa ở cuối)."""
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None); continue
        out.append(node.val); q.append(node.left); q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

T = [3, 5, 1, 6, 2, 0, 8, None, None, 7, 4]
examples = [((T, 5, 0), 3), ((T, 5, 7), 2), ((T, 5, 5), 0)]
root = build_tree(T)
print("cây dựng lại:", to_level_list(root))
for (arr, p, q), want in examples:
    print(f"p={p}, q={q} -> kỳ vọng {want}")

cây dựng lại: [3, 5, 1, 6, 2, 0, 8, None, None, 7, 4]
p=5, q=0 -> kỳ vọng 3
p=5, q=7 -> kỳ vọng 2
p=5, q=5 -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Duyệt cây bằng stack (DFS lặp)

Đệ quy trên cây sâu `10⁴` sẽ vượt giới hạn mặc định (~1000). Dùng stack tường minh.

In [2]:
def iter_nodes(root):
    st = [root]
    while st:
        node = st.pop()
        if node:
            yield node
            st.append(node.right); st.append(node.left)

print([nd.val for nd in iter_nodes(root)], "(tiền thứ tự)")

[3, 5, 6, 2, 7, 4, 1, 0, 8] (tiền thứ tự)


### 3.2 Độ sâu và con trỏ cha

BFS từ gốc, ghi `parent[node]` và `depth[node]` (gốc có độ sâu 0, cha `None`).

In [3]:
def parents_and_depths(root):
    parent = {root: None}; depth = {root: 0}; q = deque([root])
    while q:
        u = q.popleft()
        for v in (u.left, u.right):
            if v:
                parent[v] = u; depth[v] = depth[u] + 1; q.append(v)
    return parent, depth

par, dep = parents_and_depths(root)
print({nd.val: (par[nd].val if par[nd] else None, dep[nd]) for nd in par})

{3: (None, 0), 5: (3, 1), 1: (3, 1), 6: (5, 2), 2: (5, 2), 0: (1, 2), 8: (1, 2), 7: (2, 3), 4: (2, 3)}


### 3.3 Tổ tiên chung thấp nhất (LCA)

LCA của `p` và `q` là nút **sâu nhất** là tổ tiên của cả hai (một nút được coi là tổ tiên của chính nó). Đường đi duy nhất giữa `p` và `q` trong cây đi **qua LCA**.

### 3.4 Đường đi gốc → nút

Lưu stack `(nút, đường đi)` — hoặc dùng con trỏ cha rồi đảo ngược.

In [4]:
def path_to(root, target):
    parent, _ = parents_and_depths(root)
    node = next(nd for nd in parent if nd.val == target)
    path = []
    while node:
        path.append(node.val); node = parent[node]
    return path[::-1]

print(path_to(root, 7), path_to(root, 0))

[3, 5, 2, 7] [3, 1, 0]


## 4. Ý tưởng nền tảng / trực giác

Cây ví dụ:
```
            3
          /   \
         5     1
        / \   / \
       6   2 0   8
          / \
         7   4
```
- `p = 5`, `q = 0`: đường gốc→5 là `[3, 5]`, gốc→0 là `[3, 1, 0]`. Phần chung `[3]` ⇒ LCA = 3. Khoảng cách `= (2 − 1) + (3 − 1) = 1 + 2 = 3`.
- `p = 5`, `q = 7`: `[3, 5]` và `[3, 5, 2, 7]`, phần chung `[3, 5]` ⇒ LCA = 5 ⇒ `0 + 2 = 2`.

Công thức với độ sâu: `depth(5) = 1`, `depth(0) = 2`, `depth(LCA = 3) = 0` ⇒ `1 + 2 − 0 = 3`.

In [5]:
for (arr, p, q), want in examples:
    pp, pq = path_to(root, p), path_to(root, q)
    common = 0
    while common < min(len(pp), len(pq)) and pp[common] == pq[common]:
        common += 1
    print(f"p={p}: {pp} | q={q}: {pq} | chung={common} -> {len(pp) + len(pq) - 2 * common} (kỳ vọng {want})")

p=5: [3, 5] | q=0: [3, 1, 0] | chung=1 -> 3 (kỳ vọng 3)
p=5: [3, 5] | q=7: [3, 5, 2, 7] | chung=2 -> 2 (kỳ vọng 2)
p=5: [3, 5] | q=5: [3, 5] | chung=2 -> 0 (kỳ vọng 0)


## 5. Cách 1 — Brute Force / baseline: đồ thị vô hướng + BFS

**Ý tưởng:** quên cấu trúc cây; coi mỗi cạnh cha–con là cạnh vô hướng, BFS từ nút `p` tới khi gặp nút `q`. Không cần biết LCA.

In [6]:
def dist_bfs_graph(root, p, q):
    adj = {}
    for u in iter_nodes(root):
        adj.setdefault(u.val, [])
        for v in (u.left, u.right):
            if v:
                adj[u.val].append(v.val)
                adj.setdefault(v.val, []).append(u.val)
    dist = {p: 0}; dq = deque([p])
    while dq:
        u = dq.popleft()
        if u == q:
            return dist[u]
        for v in adj[u]:
            if v not in dist:
                dist[v] = dist[u] + 1; dq.append(v)
    return -1

for (arr, p, q), want in examples:
    print(dist_bfs_graph(build_tree(arr), p, q), "kỳ vọng", want)

3 kỳ vọng 3
2 kỳ vọng 2
0 kỳ vọng 0


**Độ phức tạp:** Time `O(n)` (dựng đồ thị + BFS), Space `O(n)` cho danh sách kề. Đã tối ưu về bậc, nhưng tốn bộ nhớ và chưa khai thác cấu trúc cây.

Để kiểm chứng **độc lập**, ta dùng thêm Floyd–Warshall (khoảng cách mọi cặp, `O(n³)`) trên cây nhỏ:

In [7]:
def all_pairs_floyd(root):
    nodes = [nd.val for nd in iter_nodes(root)]
    idx = {v: i for i, v in enumerate(nodes)}; n = len(nodes); INF = float("inf")
    d = [[0 if i == j else INF for j in range(n)] for i in range(n)]
    for u in iter_nodes(root):
        for v in (u.left, u.right):
            if v:
                d[idx[u.val]][idx[v.val]] = d[idx[v.val]][idx[u.val]] = 1
    for k in range(n):
        dk = d[k]
        for i in range(n):
            dik = d[i][k]
            if dik == INF: continue
            di = d[i]
            for j in range(n):
                if dik + dk[j] < di[j]:
                    di[j] = dik + dk[j]
    return idx, d

idx, D = all_pairs_floyd(root)
print("Floyd: dist(5, 0) =", D[idx[5]][idx[0]], "| dist(6, 8) =", D[idx[6]][idx[8]])

Floyd: dist(5, 0) = 3 | dist(6, 8) = 4


## 6. Ý tưởng thuật toán / bước đột phá

**Insight (cấu trúc cây):** giữa hai nút của cây có **đúng một** đường đi đơn, và nó đi qua LCA. Do đó

`dist(p, q) = depth(p) + depth(q) − 2 · depth(LCA)`.

**Ba cách tìm LCA:**
1. **Đường đi gốc→nút:** LCA là phần tử cuối của **tiền tố chung** của hai đường.
2. **Con trỏ cha + độ sâu:** đưa nút sâu hơn lên cùng độ sâu với nút kia, rồi cùng leo cho tới khi gặp nhau; số bước leo chính là khoảng cách.
3. **Hậu thứ tự:** với mỗi nút, biết khoảng cách từ nó xuống `p` (nếu `p` nằm trong cây con) và xuống `q`. Nút đầu tiên (thấp nhất) có cả hai là LCA; đáp án là tổng hai khoảng cách.

**Quy tắc nếu… thì… (cho cách 3):** tại nút `u`:
- nếu `u.val == p` thì `dp = 0`; nếu cây con trái/phải chứa `p` ở khoảng cách `x` thì `dp = x + 1`;
- tương tự `dq`;
- nếu cả `dp` và `dq` đã biết ⇒ trả về `dp + dq` (u là LCA).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hai đường đi gốc→nút, bỏ tiền tố chung

In [8]:
def root_path(root, target):
    """Đường đi gốc -> nút có giá trị target (danh sách nút), DFS lặp lưu cha."""
    parent = {root: None}; st = [root]; found = None
    while st:
        u = st.pop()
        if u.val == target:
            found = u; break
        for v in (u.left, u.right):
            if v:
                parent[v] = u; st.append(v)
    path = []
    while found:
        path.append(found); found = parent[found]
    return path[::-1]

def dist_paths(root, p, q):
    a, b = root_path(root, p), root_path(root, q)
    common = 0
    while common < len(a) and common < len(b) and a[common] is b[common]:
        common += 1
    return (len(a) - common) + (len(b) - common)

for (arr, p, q), want in examples:
    print(dist_paths(build_tree(arr), p, q), "kỳ vọng", want)

3 kỳ vọng 3
2 kỳ vọng 2
0 kỳ vọng 0


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (đường đi dài tối đa `h` — chiều cao, nhưng `parent` lưu tới `O(n)`).

### 7.2 Cách 3 — Con trỏ cha + độ sâu, leo lên gặp nhau

Từng đoạn:
1. BFS một lần: `parent`, `depth`, và tìm hai nút `x` (giá trị `p`), `y` (giá trị `q`).
2. Nếu `depth[x] < depth[y]` thì đổi `x, y`.
3. Leo `x` lên `depth[x] − depth[y]` bước (đếm vào `steps`).
4. Trong khi `x is not y`: leo cả hai, `steps += 2`.

In [9]:
def dist_parent_climb(root, p, q):
    parent = {root: None}; depth = {root: 0}; x = y = None
    dq = deque([root])
    while dq:
        u = dq.popleft()
        if u.val == p: x = u
        if u.val == q: y = u
        for v in (u.left, u.right):
            if v:
                parent[v] = u; depth[v] = depth[u] + 1; dq.append(v)
    if depth[x] < depth[y]:
        x, y = y, x
    steps = 0
    while depth[x] > depth[y]:
        x = parent[x]; steps += 1
    while x is not y:
        x, y = parent[x], parent[y]; steps += 2
    return steps

for (arr, p, q), want in examples:
    print(dist_parent_climb(build_tree(arr), p, q), "kỳ vọng", want)

3 kỳ vọng 3
2 kỳ vọng 2
0 kỳ vọng 0


**Độ phức tạp:** Time `O(n)` (BFS) + `O(h)` (leo), Space `O(n)`. Ưu điểm: nếu có **nhiều truy vấn** trên cùng cây, chỉ BFS một lần; mỗi truy vấn `O(h)` (hoặc `O(log n)` với binary lifting — xem biến thể).

### 7.3 Cách 4 — Một lượt hậu thứ tự (lặp), dừng ngay ở LCA

Mô phỏng đệ quy bằng stack hai pha: lần đầu gặp nút thì đẩy lại kèm cờ "đã xử lý con", rồi đẩy hai con; lần thứ hai thì tổng hợp thông tin từ con. Mỗi nút lưu `(dp, dq)` = khoảng cách từ nút xuống `p`/`q` trong cây con (hoặc `-1`).

In [10]:
def dist_postorder(root, p, q):
    if p == q:
        return 0
    info = {}                                   # node -> (dp, dq)
    st = [(root, False)]
    while st:
        node, done = st.pop()
        if node is None:
            continue
        if not done:
            st.append((node, True))
            st.append((node.right, False))
            st.append((node.left, False))
            continue
        dp = 0 if node.val == p else -1
        dq = 0 if node.val == q else -1
        for ch in (node.left, node.right):
            if ch is not None:
                cp, cq = info.pop(ch)
                if cp != -1: dp = cp + 1
                if cq != -1: dq = cq + 1
        if dp != -1 and dq != -1:
            return dp + dq                      # node là LCA (nút thấp nhất có cả hai)
        info[node] = (dp, dq)
    return -1

for (arr, p, q), want in examples:
    print(dist_postorder(build_tree(arr), p, q), "kỳ vọng", want)

3 kỳ vọng 3
2 kỳ vọng 2
0 kỳ vọng 0


`info.pop(ch)` giải phóng thông tin của con ngay khi cha dùng xong ⇒ bộ nhớ của `info` chỉ tỉ lệ với "biên" đang xử lý.

**Độ phức tạp:** Time `O(n)` worst (dừng sớm ngay ở LCA theo thứ tự hậu thứ tự), Space `O(h)` cho stack + `info`.

## 8. TRACE từng bước

Trace Cách 3 (leo con trỏ cha) với `p = 7`, `q = 0`:

In [11]:
def trace_climb(root, p, q):
    parent = {root: None}; depth = {root: 0}; x = y = None; dq = deque([root])
    while dq:
        u = dq.popleft()
        if u.val == p: x = u
        if u.val == q: y = u
        for v in (u.left, u.right):
            if v: parent[v] = u; depth[v] = depth[u] + 1; dq.append(v)
    print(f"x = {x.val} (sâu {depth[x]}), y = {y.val} (sâu {depth[y]})")
    if depth[x] < depth[y]:
        x, y = y, x
    steps = 0
    while depth[x] > depth[y]:
        x = parent[x]; steps += 1
        print(f"  cân độ sâu: x lên {x.val} (sâu {depth[x]}), steps = {steps}")
    while x is not y:
        x, y = parent[x], parent[y]; steps += 2
        print(f"  cùng leo: x = {x.val}, y = {y.val}, steps = {steps}")
    print(f"gặp nhau tại LCA = {x.val} -> khoảng cách {steps}")
    return steps

trace_climb(root, 7, 0)

x = 7 (sâu 3), y = 0 (sâu 2)
  cân độ sâu: x lên 2 (sâu 2), steps = 1
  cùng leo: x = 5, y = 1, steps = 3
  cùng leo: x = 3, y = 3, steps = 5
gặp nhau tại LCA = 3 -> khoảng cách 5


5

Bảng tóm tắt:

| bước | x | y | steps |
|---|---|---|---|
| đầu | 7 (sâu 3) | 0 (sâu 2) | 0 |
| cân độ sâu | 2 (sâu 2) | 0 | 1 |
| cùng leo | 5 | 1 | 3 |
| cùng leo | 3 | 3 | 5 ⇒ gặp |

Kiểm tra: đường `7 − 2 − 5 − 3 − 1 − 0` có 5 cạnh. ✓

In [12]:
def trace_postorder(root, p, q):
    info = {}; st = [(root, False)]
    while st:
        node, done = st.pop()
        if node is None: continue
        if not done:
            st.append((node, True)); st.append((node.right, False)); st.append((node.left, False)); continue
        dp = 0 if node.val == p else -1; dq_ = 0 if node.val == q else -1
        for ch in (node.left, node.right):
            if ch is not None:
                cp, cq = info.pop(ch)
                if cp != -1: dp = cp + 1
                if cq != -1: dq_ = cq + 1
        print(f"hậu thứ tự tại {node.val}: dp={dp}, dq={dq_}")
        if dp != -1 and dq_ != -1:
            print(f"-> {node.val} là LCA, khoảng cách = {dp} + {dq_} = {dp + dq_}"); return dp + dq_
        info[node] = (dp, dq_)

trace_postorder(root, 6, 4)

hậu thứ tự tại 6: dp=0, dq=-1
hậu thứ tự tại 7: dp=-1, dq=-1
hậu thứ tự tại 4: dp=-1, dq=0
hậu thứ tự tại 2: dp=-1, dq=1
hậu thứ tự tại 5: dp=1, dq=2
-> 5 là LCA, khoảng cách = 1 + 2 = 3


3

## 9. Code nộp LeetCode

Dùng Cách 3 (lặp, an toàn với cây sâu `10⁴`).

In [13]:
class Solution:
    def findDistance(self, root: Optional[TreeNode], p: int, q: int) -> int:
        parent = {root: None}
        depth = {root: 0}
        x = y = None
        dq = deque([root])
        while dq:
            u = dq.popleft()
            if u.val == p:
                x = u
            if u.val == q:
                y = u
            for v in (u.left, u.right):
                if v:
                    parent[v] = u
                    depth[v] = depth[u] + 1
                    dq.append(v)
        if depth[x] < depth[y]:
            x, y = y, x
        steps = 0
        while depth[x] > depth[y]:
            x = parent[x]
            steps += 1
        while x is not y:
            x, y = parent[x], parent[y]
            steps += 2
        return steps

sol = Solution()
for (arr, p, q), want in examples:
    got = sol.findDistance(build_tree(arr), p, q)
    print(got, "OK" if got == want else "SAI")

3 OK
2 OK
0 OK


## 10. Vì sao thuật toán đúng

**Đường đi duy nhất qua LCA.** Trong cây, giữa hai đỉnh có đúng một đường đi đơn. Gọi `L = LCA(p, q)`. Đường `p → L` (đi lên) nối với `L → q` (đi xuống) là một đường đi đơn (hai nhánh chỉ chung đúng `L`, vì nếu chung thêm một đỉnh `w ≠ L` thì `w` là tổ tiên chung sâu hơn `L` — mâu thuẫn). Độ dài: `(depth(p) − depth(L)) + (depth(q) − depth(L))`.

**Cách 3.** Sau khi cân độ sâu, `x` và `y` cùng độ sâu; tổ tiên của `x` ở độ sâu `d` là duy nhất, nên `x` và `y` gặp nhau lần đầu đúng tại tổ tiên chung sâu nhất = `L`. Tổng số bước leo = `(depth(p) − depth(L)) + (depth(q) − depth(L))`. Nếu một nút là tổ tiên của nút kia thì sau bước cân độ sâu `x is y` ngay.

**Cách 2.** Tiền tố chung của hai đường gốc→nút là đường gốc→`L`; phần còn lại là hai nhánh.

**Cách 4.** Hậu thứ tự xử lý con trước cha. Nút đầu tiên có cả `dp ≠ −1` và `dq ≠ −1` là nút thấp nhất chứa cả hai trong cây con (mọi nút trong cây con của nó đã xử lý trước, và chưa nút nào thỏa) ⇒ là `L`; `dp`, `dq` đúng là khoảng cách từ `L` xuống `p`, `q`.

## 11. Phân tích Time / Space

Gọi `n ≤ 10⁴` số nút, `h` chiều cao (`h ≤ n`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Floyd (kiểm chứng) | `O(n³)` | `O(n²)` | cây ≤ 60 nút |
| 1. Đồ thị + BFS | `O(n)` | `O(n)` | không cần LCA |
| 2. Hai đường đi | `O(n)` | `O(n)` | |
| 3. Cha + độ sâu | `O(n + h)` | `O(n)` | **nộp bài**, tốt cho nhiều truy vấn |
| 4. Hậu thứ tự lặp | `O(n)` | `O(h)` | dừng sớm ở LCA |

In [14]:
def random_tree(n, rng, shape="random"):
    """Sinh cây n nút với giá trị phân biệt; shape='chain' cho cây suy biến."""
    vals = rng.sample(range(10**9), n)
    nodes = [TreeNode(v) for v in vals]
    for i in range(1, n):
        if shape == "chain":
            par = nodes[i - 1]
            if rng.random() < 0.5 and par.left is None: par.left = nodes[i]
            elif par.right is None: par.right = nodes[i]
            else: par.left = nodes[i]
        else:
            while True:
                par = nodes[rng.randrange(i)]
                side = rng.choice(("left", "right"))
                if getattr(par, side) is None:
                    setattr(par, side, nodes[i]); break
    return nodes[0], vals

rng = random.Random(1)
for shape in ["random", "chain"]:
    r, vals = random_tree(10**4, rng, shape)
    p, q = rng.choice(vals), rng.choice(vals)
    res = []
    line = f"{shape:6s} n=1e4:"
    for name, fn in [("bfs", dist_bfs_graph), ("paths", dist_paths), ("climb", dist_parent_climb),
                     ("post", dist_postorder), ("submit", Solution().findDistance)]:
        t0 = time.perf_counter(); res.append(fn(r, p, q)); line += f" {name}={time.perf_counter() - t0:.3f}s"
    assert len(set(res)) == 1
    print(line, "| khoảng cách =", res[0])

random n=1e4: bfs=0.007s paths=0.003s climb=0.004s post=0.003s submit=0.004s | khoảng cách = 18
chain  n=1e4: bfs=0.006s paths=0.002s climb=0.002s post=0.004s submit=0.002s | khoảng cách = 2442


## 12. Bộ test

Đối chiếu với Floyd–Warshall trên **mọi cặp** `(p, q)` của cây ngẫu nhiên nhỏ (gồm `p == q`), thêm cây suy biến sâu `10⁴` để chắc chắn không lỗi đệ quy.

In [15]:
fns = [dist_bfs_graph, dist_paths, dist_parent_climb, dist_postorder, lambda r, p, q: Solution().findDistance(r, p, q)]

for (arr, p, q), want in examples:
    for fn in fns:
        assert fn(build_tree(arr), p, q) == want

edge_root = build_tree([1])
for fn in fns:
    assert fn(edge_root, 1, 1) == 0
two = build_tree([1, 2])
for fn in fns:
    assert fn(two, 1, 2) == 1 and fn(two, 2, 1) == 1

rng = random.Random(2024)
pairs_checked = 0
for _ in range(60):
    n = rng.randint(1, 25)
    r, vals = random_tree(n, rng, rng.choice(["random", "chain"]))
    idx, D = all_pairs_floyd(r)
    for p in vals:
        for q in vals:
            want = D[idx[p]][idx[q]]
            for fn in fns:
                assert fn(r, p, q) == want, (fn, p, q)
            pairs_checked += 1
print(f"đã kiểm {pairs_checked} cặp (p, q) trên cây ngẫu nhiên: khớp Floyd–Warshall")

đã kiểm 12098 cặp (p, q) trên cây ngẫu nhiên: khớp Floyd–Warshall


In [16]:
# cây suy biến thành đường thẳng 10^4 nút, p ở gốc, q ở lá
vals = list(range(10**4))
nodes = [TreeNode(v) for v in vals]
for i in range(1, len(nodes)):
    nodes[i - 1].right = nodes[i]
chain_root = nodes[0]
for fn in fns:
    assert fn(chain_root, 0, 9999) == 9999 and fn(chain_root, 5000, 4000) == 1000
print("cây đường thẳng 10^4 nút: mọi cách lặp chạy không lỗi đệ quy, kết quả đúng")

cây đường thẳng 10^4 nút: mọi cách lặp chạy không lỗi đệ quy, kết quả đúng


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Cộng độ sâu `depth(p) + depth(q)` mà quên trừ `2·depth(LCA)`

In [17]:
def wrong_sum_depth(root, p, q):
    _, depth = parents_and_depths(root)
    dv = {nd.val: d for nd, d in depth.items()}
    return dv[p] + dv[q]                    # SAI: chỉ đúng khi LCA là gốc

for (arr, p, q), want in examples:
    print(f"p={p}, q={q}: sai = {wrong_sum_depth(build_tree(arr), p, q)}, đúng = {want}")

p=5, q=0: sai = 3, đúng = 3
p=5, q=7: sai = 4, đúng = 2
p=5, q=5: sai = 2, đúng = 0


**Sửa:** trừ `2·depth(LCA)`.

### Lỗi 2 — LCA đệ quy kiểu "gặp `p` hoặc `q` thì trả về ngay" nhưng tính khoảng cách sai khi một nút là tổ tiên của nút kia

Ví dụ hàm `depth_from(lca, v)` đi tìm `v` trong cây con của `lca` — nếu viết `return 1 + max(left, right)` khi cả hai là `-1` thì kết quả lệch.

In [18]:
def wrong_depth_from(node, v):
    """Khoảng cách từ node xuống v; viết sai: không phân biệt 'không tìm thấy'."""
    st = [(node, 0)]
    best = 0
    while st:
        u, d = st.pop()
        if u is None:
            continue
        if u.val == v:
            return d
        best = max(best, d)                  # SAI: trả về độ sâu lớn nhất khi không thấy
        st.append((u.left, d + 1)); st.append((u.right, d + 1))
    return best

def wrong_lca_dist(root, p, q):
    par, dep = parents_and_depths(root)
    x = next(nd for nd in par if nd.val == p); y = next(nd for nd in par if nd.val == q)
    anc = set()
    while x: anc.add(x); x = par[x]
    while y not in anc: y = par[y]
    lca = y
    return wrong_depth_from(lca, p) + wrong_depth_from(lca.left if lca.val != q else lca, q)   # SAI thêm: tìm q chỉ trong cây con trái

for (arr, p, q), want in examples:
    print(f"p={p}, q={q}: sai = {wrong_lca_dist(build_tree(arr), p, q)}, đúng = {want}")

p=5, q=0: sai = 3, đúng = 3
p=5, q=7: sai = 0, đúng = 2
p=5, q=5: sai = 0, đúng = 0


Bản lỗi gộp hai sai lầm hay gặp: (1) hàm tìm khoảng cách không trả về tín hiệu "không thấy" (`-1`) mà trả về một số bất kỳ; (2) chỉ tìm `q` trong **một** cây con của LCA. **Sửa:** tìm cả hai từ chính LCA, trả về `-1` khi không thấy — hoặc dùng Cách 3/4.

### Lỗi 3 — DFS đệ quy trên cây sâu `10⁴`

In [19]:
import sys

def depth_recursive(node, v, d=0):
    if node is None:
        return -1
    if node.val == v:
        return d
    a = depth_recursive(node.left, v, d + 1)
    return a if a != -1 else depth_recursive(node.right, v, d + 1)

try:
    print(depth_recursive(chain_root, 9999))
except RecursionError as e:
    print("RecursionError:", str(e)[:50], f"(giới hạn {sys.getrecursionlimit()})")

RecursionError: maximum recursion depth exceeded (giới hạn 1000)


**Sửa:** viết lặp (stack/queue) — hoặc `sys.setrecursionlimit` (rủi ro tràn stack C).

### Lỗi 4 — So sánh nút bằng giá trị trong khi đi lên bằng chỉ số đường đi lệch

Ở Cách 2, so `a[i].val == b[i].val` vẫn đúng vì giá trị duy nhất; nhưng nếu so **độ dài** đường đi thay vì so từng phần tử (ví dụ `common = min(len(a), len(b))`) thì sai khi hai nút ở hai nhánh khác nhau.

In [20]:
def wrong_common(root, p, q):
    a, b = root_path(root, p), root_path(root, q)
    common = min(len(a), len(b))              # SAI: giả định đường ngắn là tiền tố của đường dài
    return (len(a) - common) + (len(b) - common)

for (arr, p, q), want in examples:
    print(f"p={p}, q={q}: sai = {wrong_common(build_tree(arr), p, q)}, đúng = {want}")

p=5, q=0: sai = 1, đúng = 3
p=5, q=7: sai = 2, đúng = 2
p=5, q=5: sai = 0, đúng = 0


**Sửa:** đếm tiền tố chung thực sự bằng cách so từng phần tử.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Khoảng cách giữa hai nút trong cây" ⇒ **LCA**: `depth(p) + depth(q) − 2·depth(LCA)`.
- Nhiều truy vấn trên cùng cây ⇒ tiền xử lý `parent`/`depth` (hoặc **binary lifting** `O(log n)`/truy vấn).
- Cây tới `10⁴` nút có thể suy biến ⇒ trong Python, viết lặp.
- Không chắc thuật toán cây? Chuyển sang **đồ thị + BFS** — luôn đúng, `O(n)`.
- Họ bài: 236 *LCA of a Binary Tree*, 863 *All Nodes Distance K*, 2096 *Step-By-Step Directions From a Binary Tree Node to Another*.

## 15. Biến thể của bài

1. **Nhiều truy vấn `(p, q)`:** BFS một lần, mỗi truy vấn leo `O(h)`; tối ưu `O(log n)` bằng **binary lifting** (`up[j][v]` = tổ tiên thứ `2^j`).
2. **Trả về chính đường đi** (danh sách giá trị) từ `p` tới `q` — 2096 yêu cầu chuỗi chỉ hướng `L`/`R`/`U`.
3. **Mọi nút cách `target` đúng `k`** (863): BFS trên đồ thị cha–con.
4. **Cạnh có trọng số:** thay `depth` bằng tổng trọng số từ gốc.

In [21]:
class BinaryLifting:
    """Biến thể 1: tiền xử lý O(n log n), mỗi truy vấn khoảng cách O(log n)."""
    def __init__(self, root):
        self.depth = {}; self.idx = {}; order = []
        par = {root: None}; dq = deque([root]); self.depth[root.val] = 0
        while dq:
            u = dq.popleft(); order.append(u)
            for v in (u.left, u.right):
                if v:
                    par[v] = u; self.depth[v.val] = self.depth[u.val] + 1; dq.append(v)
        self.LOG = max(1, len(order).bit_length())
        self.up = [{u.val: (par[u].val if par[u] else u.val) for u in order}]
        for j in range(1, self.LOG):
            prev = self.up[-1]
            self.up.append({v: prev[prev[v]] for v in prev})
    def lca(self, a, b):
        if self.depth[a] < self.depth[b]: a, b = b, a
        diff = self.depth[a] - self.depth[b]
        for j in range(self.LOG):
            if diff >> j & 1: a = self.up[j][a]
        if a == b: return a
        for j in range(self.LOG - 1, -1, -1):
            if self.up[j][a] != self.up[j][b]:
                a, b = self.up[j][a], self.up[j][b]
        return self.up[0][a]
    def dist(self, a, b):
        return self.depth[a] + self.depth[b] - 2 * self.depth[self.lca(a, b)]

bl = BinaryLifting(root)
print([bl.dist(p, q) for (_, p, q), _ in examples])
rng = random.Random(5)
r, vals = random_tree(300, rng)
bl = BinaryLifting(r)
for _ in range(500):
    p, q = rng.choice(vals), rng.choice(vals)
    assert bl.dist(p, q) == dist_parent_climb(r, p, q)
print("binary lifting khớp Cách 3 trên 500 truy vấn")

[3, 2, 0]
binary lifting khớp Cách 3 trên 500 truy vấn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đồ thị + BFS | bỏ cấu trúc cây | `O(n)` | `O(n)` | an toàn, dễ nghĩ |
| 2. Hai đường đi | tiền tố chung | `O(n)` | `O(n)` | dễ giải thích |
| 3. Cha + độ sâu | leo tới LCA | `O(n + h)` | `O(n)` | **nộp bài** |
| 4. Hậu thứ tự lặp | `(dp, dq)` từ con | `O(n)` | `O(h)` | một lượt, dừng sớm |
| Binary lifting | nhảy `2^j` | `O(n log n)` + `O(log n)`/truy vấn | `O(n log n)` | nhiều truy vấn |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** trên cây ví dụ, `findDistance(root, 6, 8)` và `findDistance(root, 7, 4)`?

**Bài 2 (tính tay):** với `root = [1,2,3,4,5,null,6,7]`, `p = 7`, `q = 6`: tìm LCA, độ sâu và khoảng cách.

**Bài 3 (điền TODO):** hoàn thiện `lca_value(root, p, q)` trả về **giá trị** của LCA, dùng con trỏ cha (không đệ quy).

**Bài 4 (tự viết lại):** viết `path_between(root, p, q)` trả về danh sách giá trị trên đường đi từ `p` tới `q` (kể cả hai đầu).

**Bài 5 (mini challenge):** LeetCode 863 — trả về mọi giá trị nút cách nút `target` đúng `k` cạnh (thứ tự tăng dần). Kiểm chứng bằng Floyd–Warshall.

In [22]:
def lca_value(root, p, q):
    # TODO: parent + depth bằng BFS; cân độ sâu; leo đồng thời tới khi gặp
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `6` và `8` ở hai nhánh của gốc. (2) `depth(6) = depth(8) = 2`, LCA = 3. (3) Khoảng cách `4`; `7` và `4` là anh em ⇒ `2`.

**Bài 2:** (1) Cây: `1 → (2, 3)`, `2 → (4, 5)`, `3 → (None, 6)`, `4 → (7)`. (2) `depth(7) = 3`, `depth(6) = 2`, LCA = 1. (3) `3 + 2 − 0 = 5`.

**Bài 3:** (1) Như Cách 3. (2) Trả về `x.val` khi `x is y`. (3) Kiểm tra với Floyd: `dist = depth(p) + depth(q) − 2·depth(LCA)`.

**Bài 4:** (1) Hai đường gốc→nút. (2) Bỏ tiền tố chung, giữ LCA một lần. (3) `nhánh_p đảo ngược + [LCA] + nhánh_q`.

**Bài 5:** (1) Dựng danh sách kề cha–con. (2) BFS từ `target` tới tầng `k`. (3) Sắp kết quả.

## 19. Đáp án tham khảo

In [23]:
# Bài 1
assert Solution().findDistance(root, 6, 8) == 4 and Solution().findDistance(root, 7, 4) == 2

# Bài 2
r2 = build_tree([1, 2, 3, 4, 5, None, 6, 7])
assert Solution().findDistance(r2, 7, 6) == 5

# Bài 3
def lca_value_ans(root, p, q):
    parent = {root: None}; depth = {root: 0}; x = y = None; dq = deque([root])
    while dq:
        u = dq.popleft()
        if u.val == p: x = u
        if u.val == q: y = u
        for v in (u.left, u.right):
            if v: parent[v] = u; depth[v] = depth[u] + 1; dq.append(v)
    while depth[x] > depth[y]: x = parent[x]
    while depth[y] > depth[x]: y = parent[y]
    while x is not y: x, y = parent[x], parent[y]
    return x.val
assert lca_value_ans(root, 5, 4) == 5 and lca_value_ans(root, 7, 0) == 3

# Bài 4
def path_between(root, p, q):
    a, b = root_path(root, p), root_path(root, q)
    c = 0
    while c < len(a) and c < len(b) and a[c] is b[c]:
        c += 1
    return [nd.val for nd in reversed(a[c - 1:])] + [nd.val for nd in b[c:]]

assert path_between(root, 5, 0) == [5, 3, 1, 0] and path_between(root, 7, 4) == [7, 2, 4] and path_between(root, 5, 5) == [5]
rng = random.Random(6)
for _ in range(30):
    r, vals = random_tree(rng.randint(1, 30), rng)
    for _ in range(20):
        p, q = rng.choice(vals), rng.choice(vals)
        path = path_between(r, p, q)
        assert path[0] == p and path[-1] == q and len(path) - 1 == Solution().findDistance(r, p, q)
print("Bài 1-4: OK")

Bài 1-4: OK


In [24]:
# Bài 5 — LeetCode 863: mọi nút cách target đúng k cạnh
def distance_k(root, target, k):
    adj = {}
    for u in iter_nodes(root):
        adj.setdefault(u.val, [])
        for v in (u.left, u.right):
            if v:
                adj[u.val].append(v.val); adj.setdefault(v.val, []).append(u.val)
    seen = {target}; frontier = [target]
    for _ in range(k):
        nxt = []
        for u in frontier:
            for v in adj[u]:
                if v not in seen:
                    seen.add(v); nxt.append(v)
        frontier = nxt
    return sorted(frontier)

print("863 ví dụ (target=5, k=2):", distance_k(root, 5, 2), "(kỳ vọng [1, 4, 7])")
rng = random.Random(7)
for _ in range(30):
    r, vals = random_tree(rng.randint(1, 25), rng)
    idx, D = all_pairs_floyd(r)
    t = rng.choice(vals); k = rng.randint(0, 6)
    assert distance_k(r, t, k) == sorted(v for v in vals if D[idx[t]][idx[v]] == k)
print("Bài 5: khớp Floyd–Warshall")

863 ví dụ (target=5, k=2): [1, 4, 7] (kỳ vọng [1, 4, 7])
Bài 5: khớp Floyd–Warshall


## 20. Template nhớ nhanh + Checklist

```python
# Khoảng cách hai nút trong cây = leo con trỏ cha tới LCA (lặp, an toàn cây sâu)
parent = {root: None}; depth = {root: 0}; dq = deque([root])
while dq:                                   # BFS: cha, độ sâu, tìm x (p) và y (q)
    u = dq.popleft()
    if u.val == p: x = u
    if u.val == q: y = u
    for v in (u.left, u.right):
        if v: parent[v] = u; depth[v] = depth[u] + 1; dq.append(v)
if depth[x] < depth[y]: x, y = y, x
steps = 0
while depth[x] > depth[y]: x = parent[x]; steps += 1      # cân độ sâu
while x is not y: x, y = parent[x], parent[y]; steps += 2  # cùng leo tới LCA
return steps
```

**Checklist trước khi submit:**
- [ ] Công thức `depth(p) + depth(q) − 2·depth(LCA)` (không quên trừ).
- [ ] `p == q` ⇒ `0`.
- [ ] Một nút là tổ tiên của nút kia ⇒ vẫn đúng (LCA là chính nút đó).
- [ ] Không đệ quy sâu (cây tới `10⁴` nút, có thể suy biến).
- [ ] So sánh nút bằng `is` (hoặc giá trị, vì giá trị duy nhất).